# When is `.sindex` worth it?

A spatial index is a pre-filtering step that lets GeoPandas avoid checking every geometry pair individually. It's what powers `sjoin`, `overlay`, and `clip` under the hood. But how much faster is it really?

This example compares a naive Python loop against `.sindex.query()` on the same data, and visualizes the speedup.

In [ ]:
%matplotlib inline
import time
import numpy as np
import geopandas
import shapely
import matplotlib.pyplot as plt
from geodatasets import get_path

## Load data

We'll use NYC sub-boroughs from the `geoda nyc` dataset, and generate 1000 random points within its bounds.

In [ ]:
nyc = geopandas.read_file(get_path("geoda nyc"))

rng = np.random.default_rng(42)
minx, miny, maxx, maxy = nyc.total_bounds
n_points = 1000
points = geopandas.GeoSeries(
    shapely.points(
        rng.uniform(minx, maxx, n_points),
        rng.uniform(miny, maxy, n_points),
    ),
    crs=nyc.crs,
)

fig, ax = plt.subplots(figsize=(7, 7))
nyc.plot(ax=ax, edgecolor="black", linewidth=0.5, color="lightblue")
points.plot(ax=ax, color="red", markersize=3)
ax.set_title(f"{len(nyc)} polygons, {len(points)} points")
plt.show()

## Naive approach

Iterate over every point–polygon pair and check `shapely.intersects` directly.

In [ ]:
start = time.perf_counter()
naive_matches = 0
for pt in points:
    for poly in nyc.geometry:
        if shapely.intersects(pt, poly):
            naive_matches += 1
naive_time = time.perf_counter() - start

print(f"Naive loop: {naive_time:.4f}s ({naive_matches} matches)")

## Using the spatial index

`.sindex.query(predicate="intersects")` uses an R-tree to skip most candidates before evaluating the exact predicate — so it only runs `shapely.intersects` on a small subset of pairs.

In [ ]:
start = time.perf_counter()
sindex_result = nyc.sindex.query(points, predicate="intersects")
sindex_time = time.perf_counter() - start

print(f".sindex:    {sindex_time:.4f}s ({sindex_result.shape[1]} matches)")

## Verify results match

Both approaches should find the same number of matches.

In [ ]:
assert naive_matches == sindex_result.shape[1], "Results differ!"
print("✅ Both approaches found the same matches.")

## Visualize the speedup

In [ ]:
speedup = naive_time / sindex_time

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    ["Naive loop", ".sindex"],
    [naive_time, sindex_time],
    color=["#d9534f", "#5cb85c"],
)
ax.set_ylabel("Time (seconds)")
ax.set_title(f"Spatial index is {speedup:.0f}× faster than a naive loop")
for bar, v in zip(bars, [naive_time, sindex_time]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        v,
        f"{v:.4f}s",
        ha="center",
        va="bottom",
    )
plt.show()

## Conclusion

With 55 polygons and 1000 points, `.sindex` is already **~45× faster** than a naive loop. The gap grows even larger as dataset size increases — which is why GeoPandas uses spatial indexes automatically inside `sjoin`, `overlay`, and `clip`.

For more on how the spatial index works, see the [User Guide page on spatial indexing](../docs/user_guide/spatial_indexing.ipynb).